In [136]:
import os, sys
from pathlib import Path
# Force JAX to ignore TPU/GPU backends in this environment.
os.environ.setdefault("JAX_PLATFORMS", "cpu")
os.environ.setdefault("JAX_PLATFORM_NAME", "cpu")
import numpy as np
import scipy.optimize as sp_opt
import jax
jax.config.update("jax_enable_x64", True)
jax.config.update("jax_platform_name", "cpu")
jnp = jax.numpy
jsp = jax.scipy
random = jax.random
grad = jax.grad
jit = jax.jit
vmap = jax.vmap
import pickle
import tensorflow_probability.substrates.jax as tfp
tfpd = tfp.distributions
import optimistix as optx

In [137]:
@jit
def test_function(variables, args):
    x, y, z = variables
    a, b, c = args
    u = jnp.sin(a*x) + jnp.cos(b*y) + jnp.exp(c*z)
    v = jnp.tanh(x) + jnp.sinh(y) + jnp.sqrt(a**2 + b**2 + c**2)*jnp.cosh(z)
    w = x**2 + y**2 + z**2 - a*b*c
    return jnp.array([u, v, w])

MACHINE_EPSILON = sys.float_info.epsilon

In [138]:
a, b, c = 1.0, 2.0, 3.0
x_true, y_true, z_true = 10.0, 5.0, 0.0
u_meas, v_meas, w_meas = test_function((x_true, y_true, z_true), (a, b, c))
measured = jnp.array([u_meas, v_meas, w_meas])
print(f"Measured values: u={u_meas}, v={v_meas}, w={w_meas}")

@jit
def noneq_system(variables, args):
    return test_function(variables, args) - measured

initial_guess = (-10.0, 20.0, 1.0)
solver = optx.LevenbergMarquardt(rtol=MACHINE_EPSILON, atol=MACHINE_EPSILON)
solution = optx.root_find(noneq_system, solver, initial_guess, args=(a, b, c))
print(f"Solution: {solution.value}")
print(f"True values: {jnp.array([x_true, y_true, z_true])}")
print(f"System at solution: {noneq_system(solution.value, (a, b, c))}")
print(f"Function at solution: {test_function(solution.value, (a, b, c))}") 

Measured values: u=-0.3830926399658221, v=78.94486796044038, w=119.0
Solution: (Array(10., dtype=float64), Array(5., dtype=float64), Array(2.29470327e-17, dtype=float64))
True values: [10.  5.  0.]
System at solution: [0. 0. 0.]
Function at solution: [ -0.38309264  78.94486796 119.        ]


In [140]:
from my_distributions import skewnormal_logpdf, logskewnormal_logpdf, skewnormal_cdf, logskewnormal_cdf
from utils import import_bh_data
DEBUG = False

bh_data = import_bh_data("../data/bh_table_1.txt")
print(f"Loaded columns: {list(bh_data.keys())}")
if DEBUG:
    for key in bh_data.keys():
        try:
            print(f"{key} type: {bh_data[key][1].dtype}")
        except:
            print(f"{key} type: {type(bh_data[key][1])}")

print(bh_data["sigma_gc"])
print(bh_data["M"])
M_log_mode = jnp.log(bh_data["M"])
M_log_left_edge = jnp.log(bh_data["M"]-bh_data["dM_low"])
M_log_right_edge = jnp.log(bh_data["M"]+bh_data["dM_high"])
sigma_gc_log_mode = jnp.log(bh_data["sigma_gc"])
sigma_gc_log_left_edge = jnp.log(bh_data["sigma_gc"]-bh_data["sigma_gc_low"])
sigma_gc_log_right_edge = jnp.log(bh_data["sigma_gc"]+bh_data["sigma_gc_high"])

Loaded columns: ['Galaxy', 'M', 'dM_low', 'dM_high', 'sigma_gc', 'sigma_gc_low', 'sigma_gc_high']
[134. 186. 202. 128. 175. 312. 320. 204. 217.  69.]
[1.5e+08 8.3e+08 1.5e+08 8.0e+07 1.2e+08 1.5e+09 3.6e+09 5.7e+08 2.1e+09
 4.1e+06]


In [141]:
@jit
def logskewnormal_mode_condition(params, args):
    """Defines the function that has to be zero at the mode of the LogSkewNormal"""
    loc, scale, shape = params
    log_mode = args
    inv_scale = 1.0 / scale
    t = (log_mode - loc) * inv_scale
    return shape * jnp.exp(-0.5 * (shape*t)**2) / (jnp.sqrt(2 * jnp.pi) * scale) - t * jsp.stats.norm.cdf(shape * t) / scale - jsp.stats.norm.cdf(shape * t)

@jit
def skewnormal_nonlin_system_mode(params, args):
    loc, scale, shape = params
    log_mode, log_left_edge, log_right_edge = args
    mode_condition = logskewnormal_mode_condition(params, log_mode)
    left_edge_condition = skewnormal_cdf(log_mode, loc=loc, scale=scale, shape=shape) - skewnormal_cdf(
        log_left_edge, loc=loc, scale=scale, shape=shape) - 0.5 * jsp.special.erf(1/jnp.sqrt(2))
    right_edge_condition = skewnormal_cdf(log_right_edge, loc=loc, scale=scale, shape=shape) - skewnormal_cdf(
        log_mode, loc=loc, scale=scale, shape=shape) - 0.5 * jsp.special.erf(1/jnp.sqrt(2))
    return jnp.array([mode_condition, left_edge_condition, right_edge_condition])

@jit
def skewnormal_nonlin_system_median(params, args):
    loc, scale, shape = params
    log_median, log_left_edge, log_right_edge = args
    log_median_condition = skewnormal_cdf(log_median, loc=loc, scale=scale, shape=shape) - 0.5
    left_edge_condition = skewnormal_cdf(log_left_edge, loc=loc, scale=scale, shape=shape) - 0.5 + 0.5 * jsp.special.erf(1/jnp.sqrt(2))
    right_edge_condition = skewnormal_cdf(log_right_edge, loc=loc, scale=scale, shape=shape) - 0.5 - 0.5 * jsp.special.erf(1/jnp.sqrt(2))
    return jnp.array([log_median_condition, left_edge_condition, right_edge_condition])

initial_guess = (M_log_mode[1], 0.5*(M_log_right_edge[1]-M_log_left_edge[1]), 0.0)
solver = optx.LevenbergMarquardt(rtol=MACHINE_EPSILON, atol=MACHINE_EPSILON)
solution = optx.least_squares(skewnormal_nonlin_system_mode, solver, initial_guess, args=(M_log_mode[1], M_log_left_edge[1], M_log_right_edge[1]), max_steps=10000)
print("MODE")
print(f"Initial guess: {initial_guess}")
print(f"Solution: {solution.value}")
print(f"System at solution: {skewnormal_nonlin_system_mode(solution.value, (M_log_mode[1], M_log_left_edge[1], M_log_right_edge[1]))}")
print(f"Least square residual: {jnp.sum(skewnormal_nonlin_system_mode(solution.value, (M_log_mode[1], M_log_left_edge[1], M_log_right_edge[1]))**2)}")

solver = optx.LevenbergMarquardt(rtol=MACHINE_EPSILON, atol=MACHINE_EPSILON)
solution = optx.least_squares(skewnormal_nonlin_system_median, solver, initial_guess, args=(M_log_mode[1], M_log_left_edge[1], M_log_right_edge[1]), max_steps=10000)
print("MEDIAN")
print(f"Initial guess: {initial_guess}")
print(f"Solution: {solution.value}")
print(f"System at solution: {skewnormal_nonlin_system_median(solution.value, (M_log_mode[1], M_log_left_edge[1], M_log_right_edge[1]))}")
print(f"Least square residual: {jnp.sum(skewnormal_nonlin_system_median(solution.value, (M_log_mode[1], M_log_left_edge[1], M_log_right_edge[1]))**2)}")

MODE
Initial guess: (Array(20.53693626, dtype=float64), Array(0.22599256, dtype=float64), 0.0)
Solution: (Array(20.57199948, dtype=float64), Array(0.23655866, dtype=float64), Array(0.10225196, dtype=float64))
System at solution: [-0.01203187 -0.10273128  0.08222118]
Least square residual: 0.01745880384237991
MEDIAN
Initial guess: (Array(20.53693626, dtype=float64), Array(0.22599256, dtype=float64), 0.0)
Solution: (Array(20.29114213, dtype=float64), Array(0.36962009, dtype=float64), Array(23.54919691, dtype=float64))
System at solution: [-0.00605609  0.00308876  0.00506445]
Least square residual: 7.186534311087714e-05


In [142]:
from quadax import quadcc

def logskewnormal_mean(loc, scale, shape):
    """Mean of the skew normal distribution."""
    integrand = lambda x: jnp.exp(x + skewnormal_logpdf(x, loc=loc, scale=scale, shape=shape))
    epsabs = MACHINE_EPSILON
    epsrel = MACHINE_EPSILON
    a, b = -np.inf, np.inf
    y, info = quadcc(integrand, [a, b], epsabs=epsabs, epsrel=epsrel, full_output=True)
    #assert info.err < max(epsabs, epsrel*abs(y))
    #np.testing.assert_allclose(y, 1/4, rtol=epsrel, atol=epsabs)
    return y

def skewnormal_mean(loc, scale, shape):
    """Mean of the skew normal distribution."""
    integrand = lambda x: x*jnp.exp(skewnormal_logpdf(x, loc=loc, scale=scale, shape=shape))
    epsabs = MACHINE_EPSILON
    epsrel = MACHINE_EPSILON
    a, b = -np.inf, np.inf
    y, info = quadcc(integrand, [a, b], epsabs=epsabs, epsrel=epsrel, full_output=True)
    #assert info.err < max(epsabs, epsrel*abs(y))
    #np.testing.assert_allclose(y, 1/4, rtol=epsrel, atol=epsabs)
    return y

@jit
def skewnormal_nonlin_system_mean(params, args):
    loc, scale, shape = params
    log_mean, log_left_edge, log_right_edge = args
    mean_condition = jnp.log(logskewnormal_mean(loc, scale, shape)) - log_mean
    left_edge_condition = skewnormal_cdf(log_mean, loc=loc, scale=scale, shape=shape) - skewnormal_cdf(
        log_left_edge, loc=loc, scale=scale, shape=shape) - 0.5 * jsp.special.erf(1/jnp.sqrt(2))
    right_edge_condition = skewnormal_cdf(log_right_edge, loc=loc, scale=scale, shape=shape) - skewnormal_cdf(
        log_mean, loc=loc, scale=scale, shape=shape) - 0.5 * jsp.special.erf(1/jnp.sqrt(2))
    return jnp.array([mean_condition, left_edge_condition, right_edge_condition])

In [143]:
solver = optx.LevenbergMarquardt(rtol=MACHINE_EPSILON, atol=MACHINE_EPSILON)
solution = optx.least_squares(skewnormal_nonlin_system_mean, solver, initial_guess, args=(M_log_mode[1], M_log_left_edge[1], M_log_right_edge[1]), max_steps=10000)
print("MEAN")
print(f"Initial guess: {initial_guess}")
print(f"Solution: {solution.value}")
print(f"System at solution: {skewnormal_nonlin_system_mean(solution.value, (M_log_mode[1], M_log_left_edge[1], M_log_right_edge[1]))}")
print(f"Least square residual: {jnp.sum(skewnormal_nonlin_system_mean(solution.value, (M_log_mode[1], M_log_left_edge[1], M_log_right_edge[1]))**2)}")

MEAN
Initial guess: (Array(20.53693626, dtype=float64), Array(0.22599256, dtype=float64), 0.0)
Solution: (Array(20.45699146, dtype=float64), Array(0.54933372, dtype=float64), Array(-0.04229008, dtype=float64))
System at solution: [ 0.05224763 -0.2180951  -0.1568758 ]
Least square residual: 0.07490530303951728


In [146]:
@jit
def skewnormal_nonlin_system_mode_mass(params, args):
    loc, scale, shape = params
    log_mode, log_left_edge, log_right_edge = args
    mode_condition = logskewnormal_mode_condition(params, log_mode)
    interval_condition = skewnormal_cdf(log_right_edge, loc=loc, scale=scale, shape=shape) - skewnormal_cdf(
        log_left_edge, loc=loc, scale=scale, shape=shape) - jsp.special.erf(1/jnp.sqrt(2))
    equal_probability_condition = logskewnormal_logpdf(jnp.exp(log_left_edge), loc=loc, scale=scale, shape=shape) - logskewnormal_logpdf(jnp.exp(log_right_edge), loc=loc, scale=scale, shape=shape)
    return jnp.array([mode_condition, interval_condition, equal_probability_condition])

@jit
def skewnormal_nonlin_system_median_mass(params, args):
    loc, scale, shape = params
    log_median, log_left_edge, log_right_edge = args
    median_condition = skewnormal_cdf(log_median, loc=loc, scale=scale, shape=shape) - 0.5
    interval_condition = skewnormal_cdf(log_right_edge, loc=loc, scale=scale, shape=shape) - skewnormal_cdf(
        log_left_edge, loc=loc, scale=scale, shape=shape) - jsp.special.erf(1/jnp.sqrt(2))
    equal_probability_condition = logskewnormal_logpdf(jnp.exp(log_left_edge), loc=loc, scale=scale, shape=shape) - logskewnormal_logpdf(jnp.exp(log_right_edge), loc=loc, scale=scale, shape=shape)
    return jnp.array([median_condition, interval_condition, equal_probability_condition])

@jit
def skewnormal_nonlin_system_mean_mass(params, args):
    loc, scale, shape = params
    log_mean, log_left_edge, log_right_edge = args
    mean_condition = jnp.log(logskewnormal_mean(loc, scale, shape)) - log_mean
    interval_condition = skewnormal_cdf(log_right_edge, loc=loc, scale=scale, shape=shape) - skewnormal_cdf(
        log_left_edge, loc=loc, scale=scale, shape=shape) - jsp.special.erf(1/jnp.sqrt(2))
    equal_probability_condition = logskewnormal_logpdf(jnp.exp(log_left_edge), loc=loc, scale=scale, shape=shape) - logskewnormal_logpdf(jnp.exp(log_right_edge), loc=loc, scale=scale, shape=shape)
    return jnp.array([mean_condition, interval_condition, equal_probability_condition])

@jit
def skewnormal_nonlin_system_mode_quantiles(params, args):
    loc, scale, shape = params
    log_mode, log_left_edge, log_right_edge = args
    mode_condition = logskewnormal_mode_condition(params, log_mode)
    left_edge_condition = skewnormal_cdf(log_left_edge, loc=loc, scale=scale, shape=shape) - 0.5 + 0.5 * jsp.special.erf(1/jnp.sqrt(2))
    right_edge_condition = skewnormal_cdf(log_right_edge, loc=loc, scale=scale, shape=shape) - 0.5 - 0.5 * jsp.special.erf(1/jnp.sqrt(2))
    return jnp.array([mode_condition, left_edge_condition, right_edge_condition])


solver = optx.LevenbergMarquardt(rtol=MACHINE_EPSILON, atol=MACHINE_EPSILON)
solution = optx.least_squares(skewnormal_nonlin_system_mode_mass, solver, initial_guess, args=(M_log_mode[1], M_log_left_edge[1], M_log_right_edge[1]), max_steps=10000)
print("MODE MASS")
print(f"Initial guess: {initial_guess}")
print(f"Solution: {solution.value}")
print(f"System at solution: {skewnormal_nonlin_system_mode_mass(solution.value, (M_log_mode[1], M_log_left_edge[1], M_log_right_edge[1]))}")
print(f"Least square residual: {jnp.sum(skewnormal_nonlin_system_mode_mass(solution.value, (M_log_mode[1], M_log_left_edge[1], M_log_right_edge[1]))**2)}")

solver = optx.LevenbergMarquardt(rtol=MACHINE_EPSILON, atol=MACHINE_EPSILON)
solution = optx.least_squares(skewnormal_nonlin_system_median_mass, solver, initial_guess, args=(M_log_mode[1], M_log_left_edge[1], M_log_right_edge[1]), max_steps=10000)
print("MEDIAN MASS")
print(f"Initial guess: {initial_guess}")
print(f"Solution: {solution.value}")
print(f"System at solution: {skewnormal_nonlin_system_median_mass(solution.value, (M_log_mode[1], M_log_left_edge[1], M_log_right_edge[1]))}")
print(f"Least square residual: {jnp.sum(skewnormal_nonlin_system_median_mass(solution.value, (M_log_mode[1], M_log_left_edge[1], M_log_right_edge[1]))**2)}")

solver = optx.LevenbergMarquardt(rtol=MACHINE_EPSILON, atol=MACHINE_EPSILON)
solution = optx.least_squares(skewnormal_nonlin_system_mean_mass, solver, initial_guess, args=(M_log_mode[1], M_log_left_edge[1], M_log_right_edge[1]), max_steps=10000)
print("MEAN MASS")
print(f"Initial guess: {initial_guess}")
print(f"Solution: {solution.value}")
print(f"System at solution: {skewnormal_nonlin_system_mean_mass(solution.value, (M_log_mode[1], M_log_left_edge[1], M_log_right_edge[1]))}")
print(f"Least square residual: {jnp.sum(skewnormal_nonlin_system_mean_mass(solution.value, (M_log_mode[1], M_log_left_edge[1], M_log_right_edge[1]))**2)}")

solver = optx.LevenbergMarquardt(rtol=MACHINE_EPSILON, atol=MACHINE_EPSILON)
solution = optx.root_find(skewnormal_nonlin_system_mode_quantiles, solver, initial_guess, args=(M_log_mode[1], M_log_left_edge[1], M_log_right_edge[1]), max_steps=10000)
print("MODE QUANTILES")
print(f"Initial guess: {initial_guess}")
print(f"Solution: {solution.value}")
print(f"System at solution: {skewnormal_nonlin_system_mode_quantiles(solution.value, (M_log_mode[1], M_log_left_edge[1], M_log_right_edge[1]))}")
print(f"Least square residual: {jnp.sum(skewnormal_nonlin_system_mode_quantiles(solution.value, (M_log_mode[1], M_log_left_edge[1], M_log_right_edge[1]))**2)}")
    
    


MODE MASS
Initial guess: (Array(20.53693626, dtype=float64), Array(0.22599256, dtype=float64), 0.0)
Solution: (Array(20.59465835, dtype=float64), Array(0.31481118, dtype=float64), Array(0.25990262, dtype=float64))
System at solution: [ 0.12813532 -0.15613993  0.14391226]
Least square residual: 0.06150907606965238
MEDIAN MASS
Initial guess: (Array(20.53693626, dtype=float64), Array(0.22599256, dtype=float64), 0.0)
Solution: (Array(20.63906599, dtype=float64), Array(0.22355478, dtype=float64), Array(2.5984926e-05, dtype=float64))
System at solution: [-0.17611598 -0.0051392   0.03155936]
Least square residual: 0.03203924159904591
MEAN MASS
Initial guess: (Array(20.53693626, dtype=float64), Array(0.22599256, dtype=float64), 0.0)
Solution: (Array(20.72959642, dtype=float64), Array(0.57946773, dtype=float64), Array(0.03811597, dtype=float64))
System at solution: [ 0.37800699 -0.3892006   0.24366437]
Least square residual: 0.35373871836160814
MODE QUANTILES
Initial guess: (Array(20.53693626, 

In [145]:
jsp.special.erf(1/jnp.sqrt(2))

Array(0.68268949, dtype=float64, weak_type=True)

In [ ]:
@jit
def logskewnormal_nonlin_system_mode_quantiles(params, args):
    loc, scale, shape = params
    log_mode, log_left_edge, log_right_edge = args
    mode_condition = logskewnormal_mode_condition(params, log_mode)
    left_edge_condition = skewnormal_cdf(log_left_edge, loc=loc, scale=scale, shape=shape) - 0.5 + 0.5 * jsp.special.erf(1/jnp.sqrt(2))
    right_edge_condition = skewnormal_cdf(log_right_edge, loc=loc, scale=scale, shape=shape) - 0.5 - 0.5 * jsp.special.erf(1/jnp.sqrt(2))
    return jnp.array([mode_condition, left_edge_condition, right_edge_condition])

solver = optx.LevenbergMarquardt(rtol=MACHINE_EPSILON, atol=MACHINE_EPSILON)
solution = optx.root_find(skewnormal_nonlin_system_mode_quantiles, solver, initial_guess, args=(M_log_mode[1], M_log_left_edge[1], M_log_right_edge[1]), max_steps=10000)

# Batched execution

In [148]:
# Batched (vmap) solves for all i on M and sigma_gc triplets
# This cell does not modify previous code; it adds vectorized wrappers.

lm_solver_batch = optx.LevenbergMarquardt(rtol=MACHINE_EPSILON, atol=MACHINE_EPSILON)


def _initial_guess_from_triplet(log_mode, log_left_edge, log_right_edge):
    return jnp.array([
        log_mode,
        0.5 * (log_right_edge - log_left_edge),
        0.0,
    ])


def _solve_one_least_squares(system_fn, log_mode, log_left_edge, log_right_edge):
    args = (log_mode, log_left_edge, log_right_edge)
    x0 = _initial_guess_from_triplet(log_mode, log_left_edge, log_right_edge)
    sol = optx.least_squares(
        system_fn,
        lm_solver_batch,
        x0,
        args=args,
        max_steps=10000,
        throw=False,
    )
    residual_vec = system_fn(sol.value, args)
    residual_l2_sq = jnp.sum(residual_vec ** 2)
    return sol.value, residual_l2_sq


def _solve_one_root_find(system_fn, log_mode, log_left_edge, log_right_edge):
    args = (log_mode, log_left_edge, log_right_edge)
    x0 = _initial_guess_from_triplet(log_mode, log_left_edge, log_right_edge)
    sol = optx.root_find(
        system_fn,
        lm_solver_batch,
        x0,
        args=args,
        max_steps=10000,
        throw=False,
    )
    residual_vec = system_fn(sol.value, args)
    residual_l2_sq = jnp.sum(residual_vec ** 2)
    return sol.value, residual_l2_sq


# Dedicated single-item solvers (needed so system_fn is not dynamic during JIT)
def _solve_mode_ls(log_mode, log_left_edge, log_right_edge):
    return _solve_one_least_squares(skewnormal_nonlin_system_mode, log_mode, log_left_edge, log_right_edge)


def _solve_median_ls(log_mode, log_left_edge, log_right_edge):
    return _solve_one_least_squares(skewnormal_nonlin_system_median, log_mode, log_left_edge, log_right_edge)


def _solve_mean_ls(log_mode, log_left_edge, log_right_edge):
    return _solve_one_least_squares(skewnormal_nonlin_system_mean, log_mode, log_left_edge, log_right_edge)


def _solve_mode_mass_ls(log_mode, log_left_edge, log_right_edge):
    return _solve_one_least_squares(skewnormal_nonlin_system_mode_mass, log_mode, log_left_edge, log_right_edge)


def _solve_median_mass_ls(log_mode, log_left_edge, log_right_edge):
    return _solve_one_least_squares(skewnormal_nonlin_system_median_mass, log_mode, log_left_edge, log_right_edge)


def _solve_mean_mass_ls(log_mode, log_left_edge, log_right_edge):
    return _solve_one_least_squares(skewnormal_nonlin_system_mean_mass, log_mode, log_left_edge, log_right_edge)


def _solve_mode_quantiles_rf(log_mode, log_left_edge, log_right_edge):
    return _solve_one_root_find(skewnormal_nonlin_system_mode_quantiles, log_mode, log_left_edge, log_right_edge)


# JIT + VMAP batched wrappers
solve_mode_ls_batch = jit(vmap(_solve_mode_ls, in_axes=(0, 0, 0)))
solve_median_ls_batch = jit(vmap(_solve_median_ls, in_axes=(0, 0, 0)))
solve_mean_ls_batch = jit(vmap(_solve_mean_ls, in_axes=(0, 0, 0)))

solve_mode_mass_ls_batch = jit(vmap(_solve_mode_mass_ls, in_axes=(0, 0, 0)))
solve_median_mass_ls_batch = jit(vmap(_solve_median_mass_ls, in_axes=(0, 0, 0)))
solve_mean_mass_ls_batch = jit(vmap(_solve_mean_mass_ls, in_axes=(0, 0, 0)))

solve_mode_quantiles_rf_batch = jit(vmap(_solve_mode_quantiles_rf, in_axes=(0, 0, 0)))


def _run_batched_solver_with_mask(batch_solver_fn, log_mode_arr, log_left_edge_arr, log_right_edge_arr):
    n = len(log_mode_arr)
    params = jnp.full((n, 3), jnp.nan)
    residual_l2_sq = jnp.full((n,), jnp.nan)

    finite_mask = (
        jnp.isfinite(log_mode_arr)
        & jnp.isfinite(log_left_edge_arr)
        & jnp.isfinite(log_right_edge_arr)
    )

    valid_idx = np.where(np.asarray(finite_mask))[0]
    if valid_idx.size == 0:
        return params, residual_l2_sq, finite_mask

    p_valid, r_valid = batch_solver_fn(
        log_mode_arr[valid_idx],
        log_left_edge_arr[valid_idx],
        log_right_edge_arr[valid_idx],
    )
    params = params.at[valid_idx].set(p_valid)
    residual_l2_sq = residual_l2_sq.at[valid_idx].set(r_valid)
    return params, residual_l2_sq, finite_mask


def run_all_batched_triplet_solves(log_mode_arr, log_left_edge_arr, log_right_edge_arr):
    mode_params, mode_res, finite_mask = _run_batched_solver_with_mask(
        solve_mode_ls_batch, log_mode_arr, log_left_edge_arr, log_right_edge_arr
    )
    median_params, median_res, _ = _run_batched_solver_with_mask(
        solve_median_ls_batch, log_mode_arr, log_left_edge_arr, log_right_edge_arr
    )
    mean_params, mean_res, _ = _run_batched_solver_with_mask(
        solve_mean_ls_batch, log_mode_arr, log_left_edge_arr, log_right_edge_arr
    )

    mode_mass_params, mode_mass_res, _ = _run_batched_solver_with_mask(
        solve_mode_mass_ls_batch, log_mode_arr, log_left_edge_arr, log_right_edge_arr
    )
    median_mass_params, median_mass_res, _ = _run_batched_solver_with_mask(
        solve_median_mass_ls_batch, log_mode_arr, log_left_edge_arr, log_right_edge_arr
    )
    mean_mass_params, mean_mass_res, _ = _run_batched_solver_with_mask(
        solve_mean_mass_ls_batch, log_mode_arr, log_left_edge_arr, log_right_edge_arr
    )

    mode_quantiles_params, mode_quantiles_res, _ = _run_batched_solver_with_mask(
        solve_mode_quantiles_rf_batch, log_mode_arr, log_left_edge_arr, log_right_edge_arr
    )

    return {
        "finite_mask": finite_mask,
        "mode": {"params": mode_params, "residual_l2_sq": mode_res},
        "median": {"params": median_params, "residual_l2_sq": median_res},
        "mean": {"params": mean_params, "residual_l2_sq": mean_res},
        "mode_mass": {"params": mode_mass_params, "residual_l2_sq": mode_mass_res},
        "median_mass": {"params": median_mass_params, "residual_l2_sq": median_mass_res},
        "mean_mass": {"params": mean_mass_params, "residual_l2_sq": mean_mass_res},
        "mode_quantiles": {"params": mode_quantiles_params, "residual_l2_sq": mode_quantiles_res},
    }


M_batched_results = run_all_batched_triplet_solves(M_log_mode, M_log_left_edge, M_log_right_edge)
sigma_batched_results = run_all_batched_triplet_solves(sigma_gc_log_mode, sigma_gc_log_left_edge, sigma_gc_log_right_edge)

print("Batched solves complete.")
print(f"N (M) = {len(M_log_mode)}, finite triplets = {int(jnp.sum(M_batched_results['finite_mask']))}")
print(f"N (sigma_gc) = {len(sigma_gc_log_mode)}, finite triplets = {int(jnp.sum(sigma_batched_results['finite_mask']))}")
print("Example: first M solution for mode [loc, scale, shape]:")
print(M_batched_results["mode"]["params"][0])
print("Example: first sigma_gc solution for mode [loc, scale, shape]:")
print(sigma_batched_results["mode"]["params"][0])
print("Max residual_l2_sq for M / sigma_gc (mode):")
print(jnp.nanmax(M_batched_results["mode"]["residual_l2_sq"]), jnp.nanmax(sigma_batched_results["mode"]["residual_l2_sq"]))

Batched solves complete.
N (M) = 10, finite triplets = 10
N (sigma_gc) = 10, finite triplets = 10
Example: first M solution for mode [loc, scale, shape]:
[18.89782386  0.40160698  0.24786808]
Example: first sigma_gc solution for mode [loc, scale, shape]:
[4.89922792e+00 3.72751050e-02 4.13525902e-05]
Max residual_l2_sq for M / sigma_gc (mode):
0.03601568447738111 0.02827815730141437


In [ ]:
##EXPERIMENTAL CELL

solver = optx.LevenbergMarquardt(rtol=MACHINE_EPSILON, atol=MACHINE_EPSILON)

def initial_guess_from_estimators(log_mode, log_left_edge, log_right_edge):
    return jnp.array([log_mode, 0.5*(log_right_edge-log_left_edge), 0.0])

def one_root_find(system, args, initial_guess, max_steps=10000, throw=False):
    solution = optx.root_find(system, solver, initial_guess, args=args, max_steps=max_steps, throw=throw)
    residuals = system(solution.value, args)
    return solution.value, residuals, jnp.sum(residuals**2)

def one_least_squares(system, args, initial_guess, max_steps=10000, throw=False):
    solution = optx.least_squares(system, solver, initial_guess, args=args, max_steps=max_steps, throw=throw)
    residuals = system(solution.value, args)
    return solution.value, residuals, jnp.sum(residuals**2)

def solve_logskewnormal_mode_quantiles(log_mode, log_left_edge, log_right_edge):
    args = (log_mode, log_left_edge, log_right_edge)
    initvals = initial_guess_from_estimators(log_mode, log_left_edge, log_right_edge)
    return one_root_find(logskewnormal_nonlin_system_mode_quantiles, args, initvals, max_steps=10000, throw=False)

batch_solver = jit(vmap(solve_logskewnormal_mode_quantiles, in_axes=(0, 0, 0)))

def run_batched_solver(batch_solver, args):
    # Unpacking the array of the observed values
    log_mode, log_left_edge, log_right_edge = args
    params, residuals, minimized_squares = batch_solver(log_mode, log_left_edge, log_right_edge)
    return params, residuals, minimized_squares


